# Conformal STL interface intersections — visual diagnostic

This notebook visualizes the geometric STL intersections computed by the conformal geometry pipeline for the standard WAKIS cavity/shell example.

The purpose is to compare, in a 2D slice,

- the **actual STL cross-section**,
- the **primal or dual FIT grid edges**,
- the **endpoint-based interface edges**, and
- the **geometrically computed STL intersection points**.

The same visualization is produced for the **primal** and **dual** grids.

A useful visual result is that the intersection markers lie on the sliced STL curves. The plot also makes it possible to spot endpoint interfaces without a geometric hit, geometric intersections invisible to endpoint classification, and edges containing multiple intersections.


## 1. Imports and repository paths

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
import numpy as np
import pyvista as pv

from wakis.gridFIT3D import GridFIT3D


def find_repo_root():
    """Find the WAKIS repository root from the current working directory."""
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "wakis").is_dir() and (candidate / "tests" / "stl").is_dir():
            return candidate
    raise RuntimeError(
        "Could not locate the WAKIS repository root. Run this notebook from inside the WAKIS repository."
    )


ROOT = find_repo_root()
CAVITY_STL = ROOT / "tests" / "stl" / "007_vacuum_cavity.stl"
SHELL_STL = ROOT / "tests" / "stl" / "007_lossymetal_shell.stl"

print("Repository root:", ROOT)
print("Cavity STL:", CAVITY_STL)
print("Shell STL:", SHELL_STL)

## 2. Build the conformal cavity/shell grid

The resolution below is intentionally moderate. It is sufficient for a visual diagnostic while keeping the geometric line tracing reasonably fast.

`GridFIT3D(..., geometry_mode="conformal")` builds the primal and dual point classifications, region topology, endpoint-interface masks, and geometric STL intersections.


In [ ]:
stl_solids = {
    "cavity": str(CAVITY_STL),
    "shell": str(SHELL_STL),
}

stl_materials = {
    "cavity": "vacuum",
    "shell": [30.0, 1.0, 30.0],
}

# Geometry diagnostic only: both STL regions use normal treatment here.
stl_material_types = {
    "cavity": "normal",
    "shell": "normal",
}

combined_surface = pv.read(CAVITY_STL) + pv.read(SHELL_STL)
xmin, xmax, ymin, ymax, zmin, zmax = combined_surface.bounds

Nx = 20
Ny = 20
Nz = 40

grid = GridFIT3D(
    xmin,
    xmax,
    ymin,
    ymax,
    zmin,
    zmax,
    Nx,
    Ny,
    Nz,
    stl_solids=stl_solids,
    stl_materials=stl_materials,
    stl_material_types=stl_material_types,
    stl_method="voxelize_rectilinear",
    geometry_mode="conformal",
    subpixel_smoothing=False,
    stl_scale=1.0,
    stl_rotate=[0.0, 0.0, 0.0],
    stl_translate=[0.0, 0.0, 0.0],
    verbose=1,
)

assert grid.interface_intersections["primal"]
assert grid.interface_intersections["dual"]

## 3. Load the transformed STL surfaces

In [ ]:
# Use GridFIT3D.read_stl() so the plotted geometry uses exactly the
# same transformations as the geometry pipeline.
surfaces = {key: grid.read_stl(key) for key in grid.stl_solids}

## 4. Numerical summary

The endpoint topology and the exact geometry are deliberately different concepts:

- **endpoint**: the two edge endpoints have different region IDs,
- **geometric**: STL tracing found at least one intersection on the edge,
- **missing**: endpoint classification predicts an interface but tracing found none,
- **hidden**: tracing found an intersection although both endpoints have the same effective region,
- **multiple**: more than one distinct intersection position lies on one edge.


In [ ]:
def intersection_summary(grid):
    rows = []

    for grid_type in ("primal", "dual"):
        for direction in ("x", "y", "z"):
            endpoint_mask = grid.interface_edge_masks[grid_type][direction]
            data = grid.interface_intersections[grid_type][direction]
            geometric_mask = data["has_intersection"]
            multiple_mask = data["multiple_intersections"]

            rows.append(
                (
                    grid_type,
                    direction,
                    int(np.count_nonzero(endpoint_mask)),
                    int(np.count_nonzero(geometric_mask)),
                    int(np.count_nonzero(endpoint_mask & ~geometric_mask)),
                    int(np.count_nonzero(geometric_mask & ~endpoint_mask)),
                    int(np.count_nonzero(multiple_mask)),
                )
            )

    header = ("grid", "dir", "endpoint", "geometric", "missing", "hidden", "multiple")
    widths = [
        max(len(str(row[i])) for row in [header, *rows]) for i in range(len(header))
    ]

    def format_row(row):
        return "  ".join(str(value).rjust(width) for value, width in zip(row, widths))

    print(format_row(header))
    print(format_row(tuple("-" * width for width in widths)))
    for row in rows:
        print(format_row(row))


intersection_summary(grid)

## 5. Plotting helpers

The plotting functions use **all geometric hit groups**, not just the single representative intersection stored in the dense arrays. Therefore an edge with two distinct intersections produces two `×` markers.


In [ ]:
def grid_coordinates(grid_type):
    if grid_type == "primal":
        return np.asarray(grid.x), np.asarray(grid.y), np.asarray(grid.z)
    if grid_type == "dual":
        return np.asarray(grid.tx), np.asarray(grid.ty), np.asarray(grid.tz)
    raise ValueError(f"Unknown grid type: {grid_type}")


def plane_info(plane):
    definitions = {
        "xy": {
            "fixed_axis": 2,
            "plot_axes": (0, 1),
            "directions": ("x", "y"),
            "labels": ("x [m]", "y [m]"),
            "fixed_label": "z",
        },
        "xz": {
            "fixed_axis": 1,
            "plot_axes": (0, 2),
            "directions": ("x", "z"),
            "labels": ("x [m]", "z [m]"),
            "fixed_label": "y",
        },
        "yz": {
            "fixed_axis": 0,
            "plot_axes": (1, 2),
            "directions": ("y", "z"),
            "labels": ("y [m]", "z [m]"),
            "fixed_label": "x",
        },
    }
    if plane not in definitions:
        raise ValueError("plane must be 'xy', 'xz', or 'yz'")
    return definitions[plane]


def nearest_slice(grid_type, plane, requested_value):
    coordinates = grid_coordinates(grid_type)
    fixed_coordinates = coordinates[plane_info(plane)["fixed_axis"]]
    index = int(np.argmin(np.abs(fixed_coordinates - requested_value)))
    return index, float(fixed_coordinates[index])


def project_point(point, plane):
    axes = plane_info(plane)["plot_axes"]
    point = np.asarray(point, dtype=float)
    return point[list(axes)]


def slice_surface(surface, plane, value):
    info = plane_info(plane)
    origin = np.zeros(3)
    origin[info["fixed_axis"]] = value
    normal = np.zeros(3)
    normal[info["fixed_axis"]] = 1.0
    return surface.slice(origin=origin, normal=normal)


def plot_polydata_lines(ax, polydata, plane, label=None, linewidth=1.8):
    if polydata.n_cells == 0:
        return

    label_used = False
    for cell_id in range(polydata.n_cells):
        cell = polydata.get_cell(cell_id)
        points = np.asarray(cell.points, dtype=float)
        if len(points) < 2:
            continue

        projected = np.asarray([project_point(point, plane) for point in points])
        ax.plot(
            projected[:, 0],
            projected[:, 1],
            linewidth=linewidth,
            label=label if not label_used else None,
        )
        label_used = True


def edge_is_in_slice(index, plane, slice_index):
    i, j, k = index
    if plane == "xy":
        return k == slice_index
    if plane == "xz":
        return j == slice_index
    if plane == "yz":
        return i == slice_index
    raise ValueError("plane must be 'xy', 'xz', or 'yz'")


def edge_segment(grid_type, direction, index, plane):
    p0, p1, _, _ = grid._get_edge_geometry(grid_type, direction, index)
    return np.asarray([project_point(p0, plane), project_point(p1, plane)])


def collect_full_grid_segments(grid_type, plane, slice_index):
    segments = []
    for direction in plane_info(plane)["directions"]:
        shape = grid.interface_intersections[grid_type][direction][
            "has_intersection"
        ].shape
        for index in np.ndindex(shape):
            if edge_is_in_slice(index, plane, slice_index):
                segments.append(edge_segment(grid_type, direction, index, plane))
    return segments


def collect_endpoint_interface_segments(grid_type, plane, slice_index):
    segments = []
    for direction in plane_info(plane)["directions"]:
        mask = grid.interface_edge_masks[grid_type][direction]
        for raw_index in np.argwhere(mask):
            index = tuple(int(v) for v in raw_index)
            if edge_is_in_slice(index, plane, slice_index):
                segments.append(edge_segment(grid_type, direction, index, plane))
    return segments


def collect_intersection_points(grid_type, plane, slice_index):
    points = []
    hidden = []
    multiple = []

    for direction in plane_info(plane)["directions"]:
        data = grid.interface_intersections[grid_type][direction]
        endpoint_mask = grid.interface_edge_masks[grid_type][direction]

        for index, groups in data["hit_groups"].items():
            index = tuple(int(v) for v in index)
            if not edge_is_in_slice(index, plane, slice_index):
                continue

            p0, p1, _, _ = grid._get_edge_geometry(grid_type, direction, index)
            p0 = np.asarray(p0, dtype=float)
            p1 = np.asarray(p1, dtype=float)
            edge_points = []

            for group in groups:
                alpha = float(np.mean([hit["alpha"] for hit in group]))
                point = p0 + alpha * (p1 - p0)
                projected = project_point(point, plane)
                points.append(projected)
                edge_points.append(projected)

                if not endpoint_mask[index]:
                    hidden.append(projected)

            if len(groups) > 1:
                multiple.extend(edge_points)

    def as_array(values):
        if not values:
            return np.empty((0, 2), dtype=float)
        return np.asarray(values, dtype=float)

    return {
        "all": as_array(points),
        "hidden": as_array(hidden),
        "multiple": as_array(multiple),
    }

## 6. Plot one primal or dual slice

Plot semantics:

- thin grid lines: all FIT edges in the selected 2D grid plane,
- thicker segments: endpoint-based interface edges,
- STL curves: exact cross-sections of the transformed STL surfaces,
- `×`: every distinct computed geometric intersection position,
- `o`: geometric intersections hidden from endpoint classification,
- `+`: positions belonging to an edge with multiple intersections.

The `o` and `+` overlays intentionally sit on top of the `×` markers.


In [ ]:
def plot_interface_slice(
    grid_type,
    plane="xz",
    slice_value=None,
    show_full_grid=True,
    show_hidden=True,
    show_multiple=True,
    ax=None,
):
    x, y, z = grid_coordinates(grid_type)

    if slice_value is None:
        bounds = {
            "xy": (z[0], z[-1]),
            "xz": (y[0], y[-1]),
            "yz": (x[0], x[-1]),
        }[plane]
        slice_value = 0.5 * (bounds[0] + bounds[1])

    slice_index, actual_value = nearest_slice(grid_type, plane, slice_value)

    if ax is None:
        _, ax = plt.subplots(figsize=(8, 6))

    if show_full_grid:
        grid_segments = collect_full_grid_segments(grid_type, plane, slice_index)
        if grid_segments:
            ax.add_collection(
                LineCollection(
                    grid_segments,
                    linewidths=0.5,
                    alpha=0.18,
                    label="FIT grid edges",
                )
            )

    interface_segments = collect_endpoint_interface_segments(
        grid_type, plane, slice_index
    )
    if interface_segments:
        ax.add_collection(
            LineCollection(
                interface_segments,
                linewidths=1.8,
                alpha=0.75,
                label="endpoint interface edges",
            )
        )

    for key, surface in surfaces.items():
        sliced = slice_surface(surface, plane, actual_value)
        plot_polydata_lines(ax, sliced, plane, label=f"STL: {key}", linewidth=2.0)

    intersection_points = collect_intersection_points(grid_type, plane, slice_index)

    points = intersection_points["all"]
    if len(points):
        ax.scatter(
            points[:, 0],
            points[:, 1],
            marker="x",
            s=42,
            linewidths=1.4,
            label="computed intersections",
            zorder=5,
        )

    hidden = intersection_points["hidden"]
    if show_hidden and len(hidden):
        ax.scatter(
            hidden[:, 0],
            hidden[:, 1],
            marker="o",
            s=80,
            facecolors="none",
            linewidths=1.2,
            label="hidden from endpoint topology",
            zorder=6,
        )

    multiple = intersection_points["multiple"]
    if show_multiple and len(multiple):
        ax.scatter(
            multiple[:, 0],
            multiple[:, 1],
            marker="+",
            s=110,
            linewidths=1.3,
            label="multiple intersections on edge",
            zorder=7,
        )

    info = plane_info(plane)
    ax.set_xlabel(info["labels"][0])
    ax.set_ylabel(info["labels"][1])
    ax.set_title(
        f"{grid_type.capitalize()} grid — {plane.upper()} slice, "
        f"{info['fixed_label']} = {actual_value:.6g} m"
    )
    ax.set_aspect("equal")
    ax.autoscale()
    ax.margins(0.03)
    ax.legend(loc="best")

    return {
        "slice_index": slice_index,
        "slice_value": actual_value,
        "intersection_points": intersection_points,
    }

## 7. Side-by-side primal and dual view

The requested physical slice coordinate is the same for both panels. Because primal and dual grid planes are staggered, each panel uses the nearest plane available on that grid. The actual coordinate used is written in the title.


In [ ]:
def plot_primal_dual(plane="xz", slice_value=None, show_full_grid=True):
    if slice_value is None:
        center = {
            "xy": 0.5 * (zmin + zmax),
            "xz": 0.5 * (ymin + ymax),
            "yz": 0.5 * (xmin + xmax),
        }
        slice_value = center[plane]

    fig, axes = plt.subplots(1, 2, figsize=(15, 6), constrained_layout=True)

    primal_result = plot_interface_slice(
        "primal",
        plane=plane,
        slice_value=slice_value,
        show_full_grid=show_full_grid,
        ax=axes[0],
    )

    dual_result = plot_interface_slice(
        "dual",
        plane=plane,
        slice_value=slice_value,
        show_full_grid=show_full_grid,
        ax=axes[1],
    )

    fig.suptitle(f"Conformal STL intersections — {plane.upper()} view")

    return {"primal": primal_result, "dual": dual_result}

## 8. Main visual diagnostic: central XZ section

For the cavity/shell geometry, the longitudinal `XZ` section is usually the most informative first view.


In [ ]:
result_xz = plot_primal_dual(
    plane="xz",
    slice_value=0.5 * (ymin + ymax),
)
plt.show()

## 9. Optional transverse and orthogonal views

Uncomment either example to inspect the other slice orientations.


In [ ]:
# Central XY plane
# result_xy = plot_primal_dual(
#     plane="xy",
#     slice_value=0.5 * (zmin + zmax),
# )
# plt.show()


In [ ]:
# Central YZ plane
# result_yz = plot_primal_dual(
#     plane="yz",
#     slice_value=0.5 * (xmin + xmax),
# )
# plt.show()


## 10. Optional local zoom

This final cell is useful for inspecting a beam-pipe/cavity transition or another local region in more detail. Set `xlim` and `ylim` after the first run.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

plot_interface_slice(
    "primal",
    plane="xz",
    slice_value=0.5 * (ymin + ymax),
    show_full_grid=True,
    ax=ax,
)

# Example zoom limits:
# ax.set_xlim(-0.08, 0.08)
# ax.set_ylim(-0.15, 0.15)

plt.show()